In [ ]:
# | default_exp dspy_infer

# DSPy SEO Inference
> Infer SEO metadata from article content using DSPy + LiteLLM.

In [ ]:
# | export
import os, dspy
from seootter.article import Article


def configure_lm(model: str | None = None, api_key: str | None = None, api_base: str | None = None):
    """Configure the DSPy language model from explicit params or env vars.
    Env var fallbacks: LLM_MODEL, LLM_API_KEY (or GROQ_API_KEY), LLM_API_BASE.
    Default model: groq/llama-3.3-70b-versatile
    """
    from dotenv import load_dotenv
    load_dotenv()
    global _lm
    model = model or os.getenv("LLM_MODEL") or "groq/llama-3.3-70b-versatile"
    api_key = api_key or os.getenv("LLM_API_KEY") or os.getenv("GROQ_API_KEY")
    api_base = api_base or os.getenv("LLM_API_BASE") or ""
    kwargs = dict(model=model, api_key=api_key)
    if api_base:
        kwargs["api_base"] = api_base
    _lm = dspy.LM(**kwargs)
    dspy.configure(lm=_lm)


In [ ]:
# | export

class InferArticleSEO(dspy.Signature):
    """You are an SEO expert. Analyze the article and extract precise SEO metadata."""
    content: str = dspy.InputField(desc="Full article text content")
    language: str = dspy.InputField(desc="Article language code e.g. 'en', 'ar'")
    focus_keyword: str = dspy.OutputField(desc="Single most important keyword this article should rank for")
    secondary_keywords: list[str] = dspy.OutputField(desc="3-5 supporting keywords related to the topic")
    target_goal: str = dspy.OutputField(desc="One sentence: who is the target reader and what action should they take")


# | export

class PredictSchema(dspy.Signature):
    """Analyze the article and suggest relevant schema.org types with confidence scores."""
    content: str = dspy.InputField(desc="Full article text content")
    suggestions: list[str] = dspy.OutputField(desc='Schema types with confidence scores, one per line like "Article: 100", "FAQPage: 60", "Product: 30". Always include Article. Only include types that genuinely fit.')
    reasoning: str = dspy.OutputField(desc="One sentence summary of recommendations")


In [ ]:
# | export

def infer_article_seo(content: str, language: str = "en") -> dict:
    "Infer SEO metadata from article content using DSPy."
    configure_lm()
    predictor = dspy.Predict(InferArticleSEO)
    result = predictor(content=content[:4000], language=language)
    return {
        "focus_keyword": result.focus_keyword,
        "secondary_keywords": result.secondary_keywords,
        "target_goal": result.target_goal,
    }


# | export

def predict_schemas(content: str) -> dict:
    "Predict the most suitable schema type for the article content using DSPy."
    configure_lm()
    predictor = dspy.Predict(PredictSchema)
    result = predictor(content=content[:3000])
    items = []
    for s in (result.suggestions or []):
        s = s.strip()
        if ":" in s:
            name, _, score = s.partition(":")
            items.append({"type": name.strip(), "score": int(score.strip().rstrip("%"))})
        elif s:
            items.append({"type": s, "score": 100})
    return {
        "suggestions": items,
        "reasoning": result.reasoning,
    }


In [ ]:
# | export

FETCH = "::fetch::"

def get_article_content(article: Article) -> str:
    "Load article content from file or fetch from URL."
    if article.file_path != FETCH:
        try:
            with open(article.file_path) as f:
                return f.read()
        except (FileNotFoundError, OSError):
            pass
    try:
        from trafilatura import fetch_url, extract
        html = fetch_url(article.url)
        if not html:
            return ""
        return extract(html, output_format="markdown", favor_recall=True,
                       include_tables=True, include_links=False, include_images=False) or ""
    except Exception:
        return ""